# 02 - チャンピオン別分析

各メンバーのチャンピオンプールと勝率を分析する。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import yaml
from pathlib import Path

for font in ['Meiryo', 'Yu Gothic', 'MS Gothic', 'Hiragino Sans']:
    try:
        matplotlib.font_manager.findfont(font, fallback_to_default=False)
        plt.rcParams['font.family'] = font
        break
    except ValueError:
        continue
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])

DATA = Path('..') / 'data' / 'processed'
CONFIG = Path('..') / 'config' / 'settings.yaml'

with open(CONFIG, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
MEMBERS = {f'{m["game_name"]}#{m["tag_line"]}' for m in cfg['members']}

df = pd.read_csv(DATA / 'player_stats.csv', parse_dates=['gameCreation'])
df_m = df[df.apply(lambda r: f'{r["summonerName"]}#{r["tagLine"]}' in MEMBERS, axis=1)].copy()

In [ ]:
# メンバー別 チャンピオン勝率（3試合以上）
MIN_GAMES = 3

champ_stats = df_m.groupby(['summonerName', 'championName']).agg(
    games=('win', 'count'),
    wins=('win', 'sum'),
    winrate=('win', 'mean'),
    avg_kda=('kda', 'mean'),
    avg_cs=('cs', 'mean'),
    avg_damage=('totalDamageDealtToChampions', 'mean'),
).round(2)

champ_stats['winrate_pct'] = (champ_stats['winrate'] * 100).round(1)
champ_filtered = champ_stats[champ_stats['games'] >= MIN_GAMES].sort_values(
    ['summonerName', 'winrate'], ascending=[True, False]
)
champ_filtered

In [ ]:
# メンバー別 得意チャンピオン TOP5
for name, grp in champ_filtered.groupby(level=0):
    top5 = grp.head(5).reset_index()
    fig, ax = plt.subplots(figsize=(8, 3))
    colors = ['#2ecc71' if wr >= 50 else '#e74c3c' for wr in top5['winrate_pct']]
    ax.barh(top5['championName'], top5['winrate_pct'], color=colors)
    ax.set_xlabel('勝率 %')
    ax.set_title(f'{name} - 得意チャンピオン TOP5 (≥{MIN_GAMES}試合)')
    ax.axvline(x=50, color='gray', linestyle='--', alpha=0.5)

    for i, (_, row) in enumerate(top5.iterrows()):
        ax.text(row['winrate_pct'] + 1, i, f"{int(row['games'])}試合", va='center', fontsize=9)

    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# チャンピオン別 平均KDA / CS / ダメージ 比較ヒートマップ
pivot_kda = champ_filtered.reset_index().pivot_table(
    index='championName', columns='summonerName', values='avg_kda'
)

if not pivot_kda.empty:
    fig, ax = plt.subplots(figsize=(10, max(6, len(pivot_kda) * 0.4)))
    sns.heatmap(pivot_kda, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax)
    ax.set_title('チャンピオン × メンバー 平均KDA')
    plt.tight_layout()
    plt.show()